# Guardrails + Observability in RBAC RAG

**Track:** Applied Agent Engineering Foundations  
**Notebook aim:** explain how enterprise RAG should enforce **guardrails** and produce **observability data** that can be audited later.

## What this notebook demonstrates
- load policy documents and salary data from S3
- chunk policy documents into retrieval-ready units
- create embeddings with Bedrock
- retrieve relevant rows using cosine similarity
- apply **RBAC guardrails** before context reaches the LLM
- capture **observability** for the request:
  - retrieval scores
  - allow/block decisions
  - model used
  - latency
  - token usage
  - answer preview

## Simple mental model

**Retrieval** finds relevant rows.  
**Guardrails** decide which of those rows are allowed to reach the model.  
**Observability** records what happened at each stage.


## Classroom framing

A basic RAG demo asks: **"What is relevant?"**

An enterprise RAG system must also ask:
- **"What is allowed?"**
- **"What evidence reached the model?"**
- **"Can we explain the decision later?"**

That is why this notebook is built around two ideas:

### Guardrails
Guardrails are control rules placed around the RAG flow.  
In this notebook, the guardrail is **RBAC**:
- L1-L4 → can see only their own salary
- L5-L6 → can see their own salary and salaries below their level
- L7 → can see all salaries

### Observability
Observability means we record what the system did.  
In this notebook, we log:
- which rows were retrieved
- which rows were blocked
- why they were blocked
- what context was finally sent to the LLM
- how long the model call took
- how many tokens were used


In [1]:
# Uncomment only if your SageMaker image is missing dependencies.
# %pip install -q boto3 pandas numpy python-docx

## Step 1 — Configuration

In [17]:
from __future__ import annotations

import io
import json
import os
import re
import time

import boto3
import numpy as np
import pandas as pd
from docx import Document as DocxDocument

# Read AWS region from the notebook environment
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# LLM model for answer generation. Available models for this class
"""
amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
amazon.nova-pro-v1:0
"""
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"


# Embedding model. Available embedding model for this class
"""
amazon.titan-embed-text-v1 
amazon.titan-embed-text-v2:0
amazon.titan-embed-image-v1
"""
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v1"

# S3 bucket
S3_BUCKET = "s3-demo-bucket-adp"
S3_PREFIX = "rag-docs/"
SALARY_KEY = "small_company_20_employees_7_roles_salary.csv"

# Retrieval settings
TOP_K = 4
OVERFETCH_K = 12
MAX_CHARS_PER_CHUNK = 350

## Step 2 — Load source data from S3

This notebook uses two data sources:

1. **Policy documents** from S3  
   These are used for normal HR policy questions.

2. **Salary roster CSV** from S3  
   These rows are sensitive, so guardrails will control access to them.


In [18]:
# Create AWS clients
s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# Read one CSV file from S3 into a DataFrame
def read_csv_from_s3(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_csv(io.BytesIO(body))

# List all policy files under the given S3 prefix
# This notebook keeps the logic simple and reads .docx policy files
def list_policy_keys(bucket: str, prefix: str) -> list[str]:
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    keys = []
    for obj in response.get("Contents", []):
        key = obj["Key"]
        if key.lower().endswith(".docx"):
            keys.append(key)
    return sorted(keys)

# Read a .docx file from S3 and convert it into plain text
def read_docx_from_s3(bucket: str, key: str) -> str:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    doc = DocxDocument(io.BytesIO(body))
    paragraphs = []

    for p in doc.paragraphs:
        text = p.text.strip()
        if text:
            paragraphs.append(text)

    return "\n\n".join(paragraphs)

# Load policy documents
policy_keys = list_policy_keys(S3_BUCKET, POLICY_PREFIX)

policy_docs = []
for key in policy_keys:
    policy_docs.append({
        "source": key,
        "text": read_docx_from_s3(S3_BUCKET, key)
    })

# Load salary roster
salary_df = read_csv_from_s3(S3_BUCKET, SALARY_KEY)

print("Policy documents loaded from S3:")
display(pd.DataFrame(policy_docs))

print("Salary roster loaded from S3:")
display(salary_df.head())

Policy documents loaded from S3:


,source,text
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...
1,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
2,rag-docs/HR-POL-003_Health_Benefits_Policy.docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
3,rag-docs/HR-POL-004_Training_and_Development_P...,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
4,rag-docs/HR-POL-005_Termination_and_Suspension...,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
5,rag-docs/HR-POL-006_Performance_Appraisal_Poli...,ABC Solutions Pvt. Ltd.\n\nDocument ID: HR-POL...
6,rag-docs/HR-POL-007_General_Work_Policy.docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
7,rag-docs/HR-POL-008_Leave_Policy.docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
8,rag-docs/HR-POL-009_Remote_Work_Policy.docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...


Salary roster loaded from S3:


,employee_id,employee_name,level,role,monthly_salary_inr,annual_salary_inr
0,EMP-1001,Aarav Sharma,L1,Associate Engineer,35000,420000
1,EMP-1002,Diya Nair,L1,Associate Engineer,36000,432000
2,EMP-1003,Ishaan Gupta,L1,Associate Engineer,34000,408000
3,EMP-1004,Meera Reddy,L2,Engineer,50000,600000
4,EMP-1005,Rohan Iyer,L2,Engineer,52000,624000


## Step 3 — Chunk policy documents

Why do we chunk?

A long document is usually too large to send as-is.  
So we split it into smaller retrieval units called **chunks**.

This notebook uses a simple **paragraph-first chunking** strategy:
- first split by blank lines
- if a paragraph is too large, split by sentence


In [19]:
# Split a policy document into retrieval-ready chunks
def semantic_like_chunks(text: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> list[str]:
    # Split the document into paragraphs
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    chunks = []
    current = ""

    for para in paragraphs:
        candidate = f"{current}\n\n{para}".strip() if current else para

        # Keep adding paragraphs until the chunk becomes too large
        if len(candidate) <= max_chars:
            current = candidate
        else:
            # Save the current chunk before starting a new one
            if current:
                chunks.append(current)

            # If one paragraph is already short enough, keep it as a chunk
            if len(para) <= max_chars:
                current = para

            # If one paragraph is too long, split it further into sentences
            else:
                sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", para) if s.strip()]
                temp = ""

                for sentence in sentences:
                    sentence_candidate = f"{temp} {sentence}".strip() if temp else sentence

                    if len(sentence_candidate) <= max_chars:
                        temp = sentence_candidate
                    else:
                        if temp:
                            chunks.append(temp)
                        temp = sentence

                current = temp

    # Add the final chunk
    if current:
        chunks.append(current)

    return chunks

# Create policy chunks
policy_chunk_rows = []

for doc in policy_docs:
    chunks = semantic_like_chunks(doc["text"])

    for i, chunk in enumerate(chunks, start=1):
        policy_chunk_rows.append({
            "doc_type": "policy",
            "source": doc["source"],
            "doc_id": doc["source"],
            "chunk_id": f'{doc["source"]}::chunk_{i:03d}',
            "text": chunk,
            "employee_id": None,
            "level_num": None
        })

# Create one salary row per employee
# Each salary row behaves like one retrieval unit
salary_chunk_rows = []

for row in salary_df.itertuples():
    level_num = int(re.findall(r"\d+", row.level)[0])

    salary_chunk_rows.append({
        "doc_type": "salary",
        "source": "salary_roster.csv",
        "doc_id": row.employee_id,
        "chunk_id": f"{row.employee_id}::salary",
        "text": f"Employee ID: {row.employee_id}. Employee Name: {row.employee_name}. Level: {row.level}. Role: {row.role}. Annual Salary INR: {int(row.annual_salary_inr)}.",
        "employee_id": row.employee_id,
        "level_num": level_num
    })

# Combine policy chunks and salary rows into one retrieval corpus
corpus_df = pd.DataFrame(policy_chunk_rows + salary_chunk_rows)

print("Combined retrieval corpus:")
display(corpus_df.tail(20))
print("Total retrieval rows:", len(corpus_df))

Combined retrieval corpus:


,doc_type,source,doc_id,chunk_id,text,employee_id,level_num
174,salary,salary_roster.csv,EMP-1001,EMP-1001::salary,Employee ID: EMP-1001. Employee Name: Aarav Sh...,EMP-1001,1.0
175,salary,salary_roster.csv,EMP-1002,EMP-1002::salary,Employee ID: EMP-1002. Employee Name: Diya Nai...,EMP-1002,1.0
176,salary,salary_roster.csv,EMP-1003,EMP-1003::salary,Employee ID: EMP-1003. Employee Name: Ishaan G...,EMP-1003,1.0
177,salary,salary_roster.csv,EMP-1004,EMP-1004::salary,Employee ID: EMP-1004. Employee Name: Meera Re...,EMP-1004,2.0
178,salary,salary_roster.csv,EMP-1005,EMP-1005::salary,Employee ID: EMP-1005. Employee Name: Rohan Iy...,EMP-1005,2.0
179,salary,salary_roster.csv,EMP-1006,EMP-1006::salary,Employee ID: EMP-1006. Employee Name: Kavya Me...,EMP-1006,2.0
180,salary,salary_roster.csv,EMP-1007,EMP-1007::salary,Employee ID: EMP-1007. Employee Name: Aditya R...,EMP-1007,2.0
181,salary,salary_roster.csv,EMP-1008,EMP-1008::salary,Employee ID: EMP-1008. Employee Name: Sneha Ku...,EMP-1008,3.0
182,salary,salary_roster.csv,EMP-1009,EMP-1009::salary,Employee ID: EMP-1009. Employee Name: Arjun Ve...,EMP-1009,3.0
183,salary,salary_roster.csv,EMP-1010,EMP-1010::salary,Employee ID: EMP-1010. Employee Name: Pooja Bh...,EMP-1010,3.0


Total retrieval rows: 194


## Step 4 — Create embeddings

Embeddings convert text into vectors.  
Once we have vectors, we can compare query and chunk similarity using cosine similarity.

### Observability note
This notebook also records:
- number of rows embedded
- embedding model used
- embedding dimension
- embedding creation time


In [20]:
# Convert one text string into an embedding vector using Bedrock
# Also return input token count for observability
def embed_text_bedrock(text: str):
    response = bedrock_runtime.invoke_model(
        modelId=BEDROCK_EMBEDDING_MODEL_ID,
        body=json.dumps({"inputText": text}),
        contentType="application/json",
        accept="application/json",
    )

    # Read JSON response
    body = json.loads(response["body"].read())

    # Return embedding vector and token count
    return body["embedding"], body.get("inputTextTokenCount")

# Normalize vectors so cosine similarity becomes a simple dot product
def normalize_matrix(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / norms

# Measure embedding stage time for observability
embedding_start_time = time.time()

# Create embeddings for all retrieval rows
corpus_embeddings = []
embedding_token_counts = []

for text in corpus_df["text"]:
    embedding, token_count = embed_text_bedrock(text)
    corpus_embeddings.append(embedding)
    embedding_token_counts.append(token_count)

# Build normalized embedding matrix
embedding_matrix = np.array(corpus_embeddings, dtype="float32")
embedding_matrix = normalize_matrix(embedding_matrix)

# Total time taken for embedding stage
embedding_seconds = round(time.time() - embedding_start_time, 3)

# Store token count for each row
corpus_df["embedding_input_tokens"] = embedding_token_counts

# Create embedding-stage observability table
embedding_observability_df = pd.DataFrame([{
    "stage": "embedding_creation",
    "model_id": BEDROCK_EMBEDDING_MODEL_ID,
    "rows_embedded": len(corpus_df),
    "embedding_dimension": embedding_matrix.shape[1],
    "total_input_tokens": sum([t for t in embedding_token_counts if t is not None]),
    "latency_seconds": embedding_seconds
}])

display(embedding_observability_df)

,stage,model_id,rows_embedded,embedding_dimension,total_input_tokens,latency_seconds
0,embedding_creation,amazon.titan-embed-text-v1,194,1536,11159,26.063


## Step 5 — Guardrails (RBAC) + retrieval audit

This is the heart of the notebook.

### What the guardrail does
The retriever may find relevant salary rows.  
But relevance alone is not enough.

Before those rows reach the LLM, the guardrail checks:
- who is asking
- what level they are
- which salary rows they are allowed to see

### What the observability table shows
The retrieval audit table records:
- retrieval rank
- similarity score
- document type
- employee id
- allow/block decision
- reason for the decision


In [26]:
# Convert a level label such as L5 into the number 5
def parse_level(level: str) -> int:
    return int(re.findall(r"\d+", level)[0])

# Standardize employee IDs such as emp-1006 -> EMP-1006
def normalize_employee_id(text: str | None) -> str | None:
    if text is None:
        return None
    return str(text).strip().upper()

# Read requester level from the salary roster
def get_requester_level(requester_employee_id: str) -> int:
    requester_id = normalize_employee_id(requester_employee_id)
    requester_row = salary_df.loc[
        salary_df["employee_id"].astype(str).str.upper() == requester_id
    ]
    return parse_level(requester_row.iloc[0]["level"])

# Find employee IDs mentioned in the question
# Also support 'my salary' by mapping it to the requester ID
def extract_requested_employee_ids(query: str, requester_employee_id: str | None = None) -> list[str]:
    ids = re.findall(r"EMP-\d{4}", query.upper())

    if not ids and requester_employee_id and "my" in query.lower():
        ids = [normalize_employee_id(requester_employee_id)]

    return list(dict.fromkeys(ids))

# Check whether the question is about salary
def is_salary_question(query: str) -> bool:
    return bool(re.search(r"\b(salary|pay|compensation|ctc)\b", query.lower()))

# Compute the set of salary rows the requester is allowed to see
def allowed_salary_ids(requester_level: int, requester_employee_id: str) -> set[str]:
    requester_id = normalize_employee_id(requester_employee_id)

    # L7 can see everyone
    if requester_level >= 7:
        return set(salary_df["employee_id"].astype(str).str.upper().tolist())

    # L1-L4 can see only their own row
    if requester_level <= 4:
        return {requester_id}

    # L5-L6 can see their own row and all rows below their level
    allowed = set(
        salary_df.loc[
            salary_df["level"].apply(parse_level) < requester_level,
            "employee_id"
        ].astype(str).str.upper().tolist()
    )
    allowed.add(requester_id)
    return allowed

# Build retrieval audit table and apply access guardrails
def build_retrieval_audit_table(query: str, requester_employee_id: str | None, top_k: int = TOP_K):
    
    # Create query embedding and capture query token count
    query_embedding, query_token_count = embed_text_bedrock(query)
    query_vector = np.array([query_embedding], dtype="float32")
    query_vector = normalize_matrix(query_vector)

    # Score all corpus rows against the query
    scores = embedding_matrix @ query_vector.T
    scores = scores.flatten()

    # Create a working copy of corpus and attach similarity scores
    retrieval_df = corpus_df.copy()
    retrieval_df["similarity_score"] = scores

    # Keep only top_k most relevant rows
    retrieval_df = retrieval_df.sort_values("similarity_score", ascending=False).head(top_k).reset_index(drop=True)

    # Apply access guardrail decision for each retrieved row
    decision_rows = []

    for row in retrieval_df.itertuples(index=False):
        is_salary_doc = row.doc_type == "salary"
        is_same_employee = row.employee_id == requester_employee_id if requester_employee_id is not None else False

        # Default decision
        access_granted = True
        decision_reason = "Allowed"

        # Salary data can be viewed only by the same employee
        if is_salary_doc and not is_same_employee:
            access_granted = False
            decision_reason = "Blocked: salary data belongs to another employee"

        decision_rows.append({
            "chunk_id": row.chunk_id,
            "doc_type": row.doc_type,
            "employee_id": row.employee_id,
            "similarity_score": row.similarity_score,
            "access_granted": access_granted,
            "decision_reason": decision_reason,
            "text": row.text,
            "query_input_tokens": query_token_count
        })

    # Convert decision rows into audit table
    retrieval_audit_df = pd.DataFrame(decision_rows)

    # Keep only authorized rows for final context
    authorized_context_df = retrieval_audit_df[retrieval_audit_df["access_granted"] == True].reset_index(drop=True)

    # requester_level is not used in this simplified version
    requester_level = None

    return retrieval_audit_df, authorized_context_df, requester_level

In [27]:
# Provide a requester employee ID for salary questions
requester_employee_id = "EMP-1006"

# Try both kinds of questions during class
query_to_test = "What is EMP-1015 salary?"
# query_to_test = "What are the rules for remote work?"

# Run retrieval + guardrails
retrieval_audit_df, authorized_context_df, requester_level = build_retrieval_audit_table(
    query=query_to_test,
    requester_employee_id=requester_employee_id,
    top_k=TOP_K
)

print("Requester employee ID:", requester_employee_id)
print("Question:", query_to_test)

print("\nRetrieval audit table:")
display(retrieval_audit_df)

print("\nAuthorized context table:")
display(authorized_context_df[["doc_type", "chunk_id", "employee_id", "similarity_score", "decision_reason", "text"]])

Requester employee ID: EMP-1006
Question: What is EMP-1015 salary?

Retrieval audit table:


,chunk_id,doc_type,employee_id,similarity_score,access_granted,decision_reason,text,query_input_tokens
0,EMP-1010::salary,salary,EMP-1010,0.494139,False,Blocked: salary data belongs to another employee,Employee ID: EMP-1010. Employee Name: Pooja Bh...,10
1,EMP-1009::salary,salary,EMP-1009,0.482812,False,Blocked: salary data belongs to another employee,Employee ID: EMP-1009. Employee Name: Arjun Ve...,10
2,EMP-1013::salary,salary,EMP-1013,0.481494,False,Blocked: salary data belongs to another employee,Employee ID: EMP-1013. Employee Name: Rahul Se...,10
3,EMP-1014::salary,salary,EMP-1014,0.481052,False,Blocked: salary data belongs to another employee,Employee ID: EMP-1014. Employee Name: Priya Pi...,10



Authorized context table:


,doc_type,chunk_id,employee_id,similarity_score,decision_reason,text


## Step 6 — Answer generation over authorized context only

This is where the notebook becomes **secure RAG**.

The LLM sees:
- only the rows that passed the guardrail
- never the blocked rows

### Observability added here
For the generation stage, we record:
- model used
- latency
- input tokens
- output tokens
- total tokens
- number of authorized rows sent to the model


In [28]:
# Call the Bedrock LLM and return both answer text and observability data
def call_bedrock(prompt: str):
    start_time = time.time()

    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.1}
    )

    latency_seconds = round(time.time() - start_time, 3)

    answer = response["output"]["message"]["content"][0]["text"]
    usage = response.get("usage", {})

    call_observability = {
        "model_id": BEDROCK_LLM_MODEL_ID,
        "latency_seconds": latency_seconds,
        "input_tokens": usage.get("inputTokens"),
        "output_tokens": usage.get("outputTokens"),
        "total_tokens": usage.get("totalTokens")
    }

    return answer, call_observability

# Run the full secure RAG flow
def answer_secure_query(query: str, requester_employee_id: str | None):
    # First retrieve rows and apply guardrails
    retrieval_audit_df, authorized_context_df, requester_level = build_retrieval_audit_table(
        query=query,
        requester_employee_id=requester_employee_id,
        top_k=TOP_K
    )

    # Build the final context string from only authorized rows
    context = "\n\n".join(
        [f"[{i+1}] {text}" for i, text in enumerate(authorized_context_df["text"].tolist())]
    )

    # Create the final generation prompt
    prompt = f"""
You are an enterprise HR assistant using retrieval-augmented generation.

Answer only from the authorized context below.
For salary questions, mention employee ID and salary details only if they are present in context.
For policy questions, answer like a normal HR policy assistant.
If the answer is not present in the authorized context, say:
I do not know from the authorized context.

Authorized context:
{context}

Question:
{query}

Answer:
""".strip()

    # Call the LLM
    answer, llm_observability = call_bedrock(prompt)

    # Build request-level observability
    request_observability_df = pd.DataFrame([{
        "question": query,
        "requester_employee_id": requester_employee_id if requester_employee_id else "(blank)",
        "requester_level": f"L{requester_level}" if requester_level is not None else "Not needed",
        "is_salary_question": is_salary_question(query),
        "authorized_rows_sent_to_llm": len(authorized_context_df),
        "model_id": llm_observability["model_id"],
        "latency_seconds": llm_observability["latency_seconds"],
        "input_tokens": llm_observability["input_tokens"],
        "output_tokens": llm_observability["output_tokens"],
        "total_tokens": llm_observability["total_tokens"],
        "answer_preview": answer[:180]
    }])

    return retrieval_audit_df, authorized_context_df, request_observability_df, answer

# Run the full secure RAG pipeline
sample_retrieval_audit_df, sample_authorized_df, request_observability_df, sample_secure_answer = answer_secure_query(
    query=query_to_test,
    requester_employee_id=requester_employee_id
)

print("Authorized context sent to the LLM:")
display(sample_authorized_df[["doc_type", "chunk_id", "employee_id", "similarity_score", "text"]])

print("\nLLM answer:")
print(sample_secure_answer)

print("\nRequest-level observability:")
display(request_observability_df)

Authorized context sent to the LLM:


,doc_type,chunk_id,employee_id,similarity_score,text



LLM answer:
I do not know from the authorized context.

Request-level observability:


,question,requester_employee_id,requester_level,is_salary_question,authorized_rows_sent_to_llm,model_id,latency_seconds,input_tokens,output_tokens,total_tokens,answer_preview
0,What is EMP-1015 salary?,EMP-1006,Not needed,True,0,amazon.nova-lite-v1:0,0.225,92,10,102,I do not know from the authorized context.


## Step 7 — Final audit summary

The final tables let us explain the request from three angles:

1. **Retrieval audit**  
   What the retriever found and how guardrails judged each row

2. **Authorized context**  
   What the model was actually allowed to see

3. **Request observability**  
   What happened at runtime:
   - model id
   - latency
   - token usage
   - answer preview


In [30]:
# Build one compact audit summary row for the classroom
final_audit_summary_df = pd.DataFrame([{
    "question": query_to_test,
    "requester_employee_id": requester_employee_id,
    "is_salary_question": is_salary_question(query_to_test),
    "authorized_rows_sent_to_llm": len(sample_authorized_df),
    "authorized_chunk_ids": " | ".join(sample_authorized_df["chunk_id"].tolist()),
    "blocked_rows": int((sample_retrieval_audit_df["access_granted"] == False).sum()),
    "llm_model_id": request_observability_df.iloc[0]["model_id"],
    "latency_seconds": request_observability_df.iloc[0]["latency_seconds"],
    "input_tokens": request_observability_df.iloc[0]["input_tokens"],
    "output_tokens": request_observability_df.iloc[0]["output_tokens"],
    "total_tokens": request_observability_df.iloc[0]["total_tokens"]
}])

print("Embedding-stage observability:")
display(embedding_observability_df)

print("\nFinal request audit summary:")
display(final_audit_summary_df)

Embedding-stage observability:


,stage,model_id,rows_embedded,embedding_dimension,total_input_tokens,latency_seconds
0,embedding_creation,amazon.titan-embed-text-v1,194,1536,11159,26.063



Final request audit summary:


,question,requester_employee_id,is_salary_question,authorized_rows_sent_to_llm,authorized_chunk_ids,blocked_rows,llm_model_id,latency_seconds,input_tokens,output_tokens,total_tokens
0,What is EMP-1015 salary?,EMP-1006,True,0,,4,amazon.nova-lite-v1:0,0.225,92,10,102
